# News Category Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `category`

## 2 · Project Overview

We classify **news articles into 5 categories** (Politics, Sports, Technology, Business, Entertainment) using tabularized text features: word count, average word length, quotes, numbers, sentiment, entity presence, and reading level.

This demonstrates **multiclass classification** with features extracted from text.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given extracted text statistics from a news article (word count, sentiment, reading level, entity flags), classify it into one of 5 categories.

## 5 · Why This Project Matters

- **News categorization** is essential for content recommendation.
- Demonstrates multiclass classification with text-derived features.
- Shows how simple text statistics can differentiate article types.
- Foundation for understanding full NLP classification pipelines.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | word_count, avg_word_length, num_quotes, num_numbers, sentiment_score, has_person_name, has_org_name, reading_level |
| **Target** | category (5 classes: Politics, Sports, Technology, Business, Entertainment) |
| **Class balance** | Roughly equal |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "category"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: category
Save dir: E:\Github\Machine-Learning-Projects\Classification\News Category Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 articles with tabularized text features (word count, sentiment proxy, topic keywords) and 5 news categories.

In [4]:
np.random.seed(SEED)
n = 6000
categories = ["Politics", "Sports", "Technology", "Business", "Entertainment"]
category = np.random.choice(categories, n, p=[0.2, 0.25, 0.2, 0.2, 0.15])

# Simulate tabularized features that vary by category
word_count = np.zeros(n)
avg_word_length = np.zeros(n)
num_quotes = np.zeros(n)
num_numbers = np.zeros(n)
sentiment_score = np.zeros(n)
has_person_name = np.zeros(n, dtype=int)
has_org_name = np.zeros(n, dtype=int)
reading_level = np.zeros(n)

for i, cat in enumerate(category):
    if cat == "Politics":
        word_count[i] = np.random.normal(500, 100)
        avg_word_length[i] = np.random.normal(5.5, 0.5)
        num_quotes[i] = np.random.poisson(4)
        num_numbers[i] = np.random.poisson(3)
        sentiment_score[i] = np.random.normal(-0.1, 0.3)
        has_person_name[i] = np.random.binomial(1, 0.9)
        has_org_name[i] = np.random.binomial(1, 0.7)
        reading_level[i] = np.random.normal(12, 1.5)
    elif cat == "Sports":
        word_count[i] = np.random.normal(350, 80)
        avg_word_length[i] = np.random.normal(4.8, 0.4)
        num_quotes[i] = np.random.poisson(2)
        num_numbers[i] = np.random.poisson(8)
        sentiment_score[i] = np.random.normal(0.2, 0.4)
        has_person_name[i] = np.random.binomial(1, 0.95)
        has_org_name[i] = np.random.binomial(1, 0.8)
        reading_level[i] = np.random.normal(8, 1.5)
    elif cat == "Technology":
        word_count[i] = np.random.normal(450, 120)
        avg_word_length[i] = np.random.normal(5.8, 0.6)
        num_quotes[i] = np.random.poisson(1)
        num_numbers[i] = np.random.poisson(5)
        sentiment_score[i] = np.random.normal(0.1, 0.2)
        has_person_name[i] = np.random.binomial(1, 0.5)
        has_org_name[i] = np.random.binomial(1, 0.85)
        reading_level[i] = np.random.normal(13, 1.5)
    elif cat == "Business":
        word_count[i] = np.random.normal(420, 100)
        avg_word_length[i] = np.random.normal(5.3, 0.5)
        num_quotes[i] = np.random.poisson(3)
        num_numbers[i] = np.random.poisson(7)
        sentiment_score[i] = np.random.normal(0.0, 0.25)
        has_person_name[i] = np.random.binomial(1, 0.6)
        has_org_name[i] = np.random.binomial(1, 0.9)
        reading_level[i] = np.random.normal(12, 1.5)
    else:  # Entertainment
        word_count[i] = np.random.normal(300, 90)
        avg_word_length[i] = np.random.normal(4.5, 0.4)
        num_quotes[i] = np.random.poisson(3)
        num_numbers[i] = np.random.poisson(2)
        sentiment_score[i] = np.random.normal(0.3, 0.4)
        has_person_name[i] = np.random.binomial(1, 0.85)
        has_org_name[i] = np.random.binomial(1, 0.4)
        reading_level[i] = np.random.normal(7, 1.5)

df = pd.DataFrame({
    "word_count": np.round(word_count.clip(50, 1000), 0).astype(int),
    "avg_word_length": np.round(avg_word_length.clip(3, 8), 2),
    "num_quotes": num_quotes.clip(0, 15).astype(int),
    "num_numbers": num_numbers.clip(0, 20).astype(int),
    "sentiment_score": np.round(sentiment_score.clip(-1, 1), 3),
    "has_person_name": has_person_name,
    "has_org_name": has_org_name,
    "reading_level": np.round(reading_level.clip(3, 18), 1),
    "category": category
})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['category'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (6000, 9)
Class balance:
category
Sports           0.250
Politics         0.205
Business         0.199
Technology       0.198
Entertainment    0.147
Name: proportion, dtype: float64


,word_count,avg_word_length,num_quotes,num_numbers,sentiment_score,has_person_name,has_org_name,reading_level,category
0,266,4.99,2,11,1.000,1,1,11.0,Sports
1,353,4.35,2,3,-0.208,1,1,8.0,Entertainment
2,321,5.42,2,11,0.302,0,1,11.9,Business
3,438,4.54,1,4,0.077,0,1,13.4,Technology
4,549,4.53,5,3,0.328,1,1,11.4,Politics


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (6000, 9)

Missing values:
word_count         0
avg_word_length    0
num_quotes         0
num_numbers        0
sentiment_score    0
has_person_name    0
has_org_name       0
reading_level      0
category           0
dtype: int64

Duplicate rows: 0

Dtypes:
word_count           int64
avg_word_length    float64
num_quotes           int64
num_numbers          int64
sentiment_score    float64
has_person_name      int64
has_org_name         int64
reading_level      float64
category            object
dtype: object

Target 'category' confirmed. Value counts:
category
Sports           1503
Politics         1230
Business         1194
Technology       1190
Entertainment     883
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["word_count", "avg_word_length", "num_numbers", "sentiment_score", "reading_level"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by News Category", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `category`.

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("News Category Distribution")
plt.tight_layout()
plt.show()
print(f"Category counts:\n{df[TARGET].value_counts()}")

Category counts:
category
Sports           1503
Politics         1230
Business         1194
Technology       1190
Entertainment     883
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4800, 8), Test: (1200, 8)
Train class distribution:
category
3    0.251
2    0.205
0    0.199
4    0.198
1    0.147
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: Target `category` encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: Roughly balanced across 5 categories.
- **Note**: Tabularized approach — real NLP would use text embeddings.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["content_density"] = X_train["word_count"] * X_train["avg_word_length"]
X_test["content_density"] = X_test["word_count"] * X_test["avg_word_length"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['word_count', 'avg_word_length', 'num_quotes', 'num_numbers', 'sentiment_score', 'has_person_name', 'has_org_name', 'reading_level', 'content_density']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7858
F1       : 0.7840

              precision    recall  f1-score   support

           0       0.69      0.61      0.64       239
           1       0.84      0.89      0.87       177
           2       0.78      0.80      0.79       246
           3       0.86      0.87      0.87       300
           4       0.74      0.77      0.76       238

    accuracy                           0.79      1200
   macro avg       0.78      0.79      0.78      1200
weighted avg       0.78      0.79      0.78      1200



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
LogisticRegression             0.850833           0.851127  0.971647  0.850297   0.850137  0.850833    0.070218
SVC                            0.840833           0.840935       NaN  0.840385   0.840101  0.840833    0.468822
NuSVC                          0.839167           0.839071       NaN  0.838583   0.838945  0.839167    1.080301
LinearDiscriminantAnalysis     0.833333           0.834547  0.970553  0.832644   0.833209  0.833333    0.037248
CatBoostClassifier             0.830833           0.831506  0.967441  0.830493   0.830282  0.830833    5.003266
CalibratedClassifierCV         0.830833           0.830397  0.958569  0.828095   0.830253  0.830833    0.147100
GaussianNB                     0.828333           0.829189  0.966333  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: catboost
Accuracy: 0.8283
F1: 0.8280


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.8413  (5.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[102]	valid_0's multi_logloss: 0.452137
LightGBM F1: 0.8313  (4.2s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.8417  0.8413     0.8411  0.8417
LightGBM               0.8317  0.8313     0.8310  0.8317
FLAML                  0.8283  0.8280     0.8278  0.8283
Logistic Regression    0.7858  0.7840     0.7834  0.7858

Top 3 models: ['CatBoost', 'LightGBM', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  CatBoost:
    Accuracy : 0.8417
    F1       : 0.8413
    Precision: 0.8411
    Recall   : 0.8417

  LightGBM:
    Accuracy : 0.8317
    F1       : 0.8313
    Precision: 0.8310
    Recall   : 0.8317

  FLAML:
    Accuracy : 0.8283
    F1       : 0.8280
    Precision: 0.8278
    Recall   : 0.8283


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: CatBoost

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.73      0.74       239
           1       0.91      0.95      0.93       177
           2       0.82      0.82      0.82       246
           3       0.93      0.91      0.92       300
           4       0.80      0.81      0.80       238

    accuracy                           0.84      1200
   macro avg       0.84      0.84      0.84      1200
weighted avg       0.84      0.84      0.84      1200


Total errors: 190 / 1200 (15.8%)

Sample misclassifications:
      word_count  avg_word_length  num_quotes  num_numbers  sentiment_score  has_person_name  has_org_name  reading_level  content_density  actual  predicted  correct
5555         409             5.70           0            8           -0.193                0             0           13.0          2331.30       0          4    False
3208         452             4.30           2            6           -

## 25 · Interpretation and Business Insight

**Key findings:**
- **Reading level** strongly differentiates categories (Entertainment low, Technology high).
- **Number count** is high in Sports and Business articles.
- **Sentiment** varies — Entertainment is most positive, Politics most negative.
- **Word count** helps separate short Entertainment pieces from longer political articles.

**Business takeaway:** Even simple text statistics can achieve reasonable categorization. Full NLP models would improve further.

## 26 · Limitations

1. Synthetic features with exaggerated category differences.
2. No actual text content or semantic features.
3. Missing topic-specific vocabulary features.
4. 5 categories is a simplification of real news taxonomies.
5. No temporal or trending topic features.

## 27 · How to Improve This Project

1. Use actual article text with TF-IDF or embeddings.
2. Add topic model features (LDA, NMF).
3. Include headline-specific features.
4. Try ModernBERT for full text classification.
5. Add more granular categories (subcategories).

## 28 · Production Considerations

- Use as a fast pre-filter before full NLP models.
- Combine with content-based NLP for robust categorization.
- Monitor for category drift as news topics evolve.
- Update features for new article styles.
- Consider hierarchical classification for subcategories.

## 29 · Common Mistakes

1. Relying solely on text statistics for categorization.
2. Not considering overlapping categories (politics + business).
3. Using accuracy without checking per-class performance.
4. Ignoring the context and semantics of articles.
5. Not handling emerging categories or topics.

## 30 · Mini Challenge / Exercises

1. Remove `reading_level` — how much does accuracy drop?
2. Merge Politics and Business into one class and retrain.
3. Try one-vs-rest classification and compare.
4. Plot per-class F1 scores — which category is hardest?
5. Add `quote_density = num_quotes / word_count * 100` and test.

## 31 · Final Summary / Key Takeaways

1. **Reading level and word count** are the most discriminating features.
2. **Sentiment and number counts** vary systematically across categories.
3. **Multiclass tree models** handle 5-way classification well.
4. **Tabularized text features** are a fast baseline for text classification.
5. **Full NLP models** would significantly improve accuracy.
6. **Per-class analysis** reveals which categories are most confusable.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\News Category Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.8417,
    "f1": 0.8413,
    "precision": 0.8411,
    "recall": 0.8417
  },
  "LightGBM": {
    "accuracy": 0.8317,
    "f1": 0.8313,
    "precision": 0.831,
    "recall": 0.8317
  },
  "Logistic Regression": {
    "accuracy": 0.7858,
    "f1": 0.784,
    "precision": 0.7834,
    "recall": 0.7858
  },
  "FLAML": {
    "accuracy": 0.8283,
    "f1": 0.828,
    "precision": 0.8278,
    "recall": 0.8283
  }
}
